# BHM Urban Futures — Portfolio Walkthrough

This notebook demonstrates the modeling workflow:

1. load Birmingham-Hoover historical data;
2. calibrate a robust baseline;
3. run one integrated urban-system scenario;
4. allocate metro population across the seven MSA counties;
5. quantify uncertainty with Monte Carlo simulation;
6. discover vulnerability conditions with machine learning.

**Interpretation:** scenario outputs are conditional simulations, not official forecasts.

In [ ]:
from pathlib import Path
import sys
import matplotlib.pyplot as plt
import pandas as pd

ROOT = Path("..").resolve()
sys.path.insert(0, str(ROOT))

from src.model import load_history, calibrate, Scenario, simulate, run_monte_carlo
from src.spatial import load_county_history, allocate_metro_population
from src.discovery import generate_experiments, fit_vulnerability_model

In [ ]:
history = load_history(ROOT / "data" / "birmingham_msa_history.csv")
cal = calibrate(history)
cal

In [ ]:
scenario = Scenario(years=20, seed=42)
sim = simulate(cal, scenario)
sim.tail()

In [ ]:
plt.figure(figsize=(9,5))
plt.plot(sim["year"], sim["population"])
plt.title("Simulated Birmingham-Hoover population")
plt.xlabel("Year")
plt.ylabel("Population")
plt.grid(alpha=.25)
plt.show()

In [ ]:
counties = load_county_history(ROOT / "data" / "msa_county_population_2021_2025.csv")
spatial = allocate_metro_population(sim, counties, seed=42)
spatial[spatial["year"] == spatial["year"].max()].sort_values("population", ascending=False)

In [ ]:
long, summary = run_monte_carlo(cal, scenario, n=300, seed=123)

plt.figure(figsize=(9,5))
plt.fill_between(summary["year"], summary["population_p10"], summary["population_p90"], alpha=.2)
plt.plot(summary["year"], summary["population_p50"])
plt.title("Population uncertainty: median and 10th–90th percentile")
plt.xlabel("Year")
plt.ylabel("Population")
plt.grid(alpha=.25)
plt.show()

In [ ]:
experiments = generate_experiments(cal, scenario, n=500, seed=404)
rf, importance = fit_vulnerability_model(experiments, seed=404)
importance

## Portfolio interpretation

A strong discussion of this project should distinguish:

- **observed inputs** from **modeled assumptions**;
- **forecast uncertainty** from point predictions;
- **threshold detection** from normative policy recommendations;
- **metro-wide dynamics** from county-level spatial redistribution.

The next technical step would be rolling historical backtesting and replacement of indexed service/infrastructure modules with observed local administrative data.